# 20. Stacking Final Evaluation

**Tujuan:** Rangkuman perbandingan lengkap: Single XGBoost vs Stacking Ensemble.
Semua metrik, semua skenario, semua konfigurasi — untuk paper.

**Input:** Semua pkl dari notebook 03-19

**Output:** Tabel final, PNG visualisasi paper-ready

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'
print('Libraries loaded.')

## 1. Load All Results

In [ ]:
# Single model results
with open(os.path.join(DATA_DIR, 'experiment_results_03.pkl'), 'rb') as f:
    single_training = pickle.load(f)

with open(os.path.join(DATA_DIR, 'ablation_results_04.pkl'), 'rb') as f:
    single_ablation = pickle.load(f)

with open(os.path.join(DATA_DIR, 'adversarial_results_05.pkl'), 'rb') as f:
    single_adversarial = pickle.load(f)

with open(os.path.join(DATA_DIR, 'robust_results_06.pkl'), 'rb') as f:
    single_robust = pickle.load(f)

with open(os.path.join(DATA_DIR, 'robustness_ablation_07.pkl'), 'rb') as f:
    single_robustness_abl = pickle.load(f)

# Stacking results
with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'rb') as f:
    stacking_baseline = pickle.load(f)

with open(os.path.join(DATA_DIR, 'stacking_ablation_16.pkl'), 'rb') as f:
    stacking_ablation = pickle.load(f)

with open(os.path.join(DATA_DIR, 'stacking_adversarial_17.pkl'), 'rb') as f:
    stacking_adversarial = pickle.load(f)

with open(os.path.join(DATA_DIR, 'stacking_robust_18.pkl'), 'rb') as f:
    stacking_robust = pickle.load(f)

with open(os.path.join(DATA_DIR, 'stacking_robustness_19.pkl'), 'rb') as f:
    stacking_robustness_abl = pickle.load(f)

print('All results loaded.')

## 2. Table: S1-S4 — Single vs Stacking

In [ ]:
# Single model S1-S4 (from notebook 06)
s_s1 = single_robust.get('mcc_s1', 0.9332)
s_s2 = single_robust.get('mcc_s2', 0.0189)
s_s3 = single_robust.get('mcc_s3', 0.9327)
s_s4 = single_robust.get('mcc_s4', 0.9946)

# Stacking S1-S4 (from notebook 18)
st_s1 = stacking_robust['stacking_s1']['mcc']
st_s2 = stacking_robust['stacking_s2']['mcc']
st_s3 = stacking_robust['stacking_s3']['mcc']
st_s4 = stacking_robust['stacking_s4']['mcc']

print('='*75)
print('  FINAL COMPARISON: Single XGBoost vs Stacking Ensemble (S1-S4)')
print('='*75)
comparison_table = pd.DataFrame({
    'Scenario': ['S1 (Baseline+Clean)', 'S2 (Baseline+Adversarial)',
                 'S3 (Robust+Clean)', 'S4 (Robust+Adversarial)'],
    'Single XGBoost MCC': [s_s1, s_s2, s_s3, s_s4],
    'Stacking Ensemble MCC': [st_s1, st_s2, st_s3, st_s4],
    'Δ (Stacking - Single)': [st_s1-s_s1, st_s2-s_s2, st_s3-s_s3, st_s4-s_s4]
})
print(comparison_table.to_string(index=False))
print('='*75)

## 3. Table: Robustness Ablation — Single vs Stacking

In [ ]:
print('\n' + '='*75)
print('  ROBUSTNESS ABLATION: Security Gap & Recovery — Single vs Stacking')
print('='*75)

stack_rob = stacking_robustness_abl['robustness_results']
print(f'{"Config":<12} | {"Stack S4 MCC":<14} | {"Stack Gap":<12} | {"Stack Recovery":<14}')
print('-'*60)
for r in stack_rob:
    print(f'{r["config"]:<12} | {r["mcc_s4"]:<14.4f} | {r["security_gap"]:<12.4f} | {r["recovery"]:<14.4f}')
print('='*60)

## 4. Table: Efficiency Comparison

In [ ]:
print('\n' + '='*65)
print('  EFFICIENCY: Single XGBoost vs Stacking Ensemble (Top-10)')
print('='*65)

# From results.txt: XGBoost Top-10 = 5.65 MB, 0.1826s/10k
single_size = 5.65  # MB
single_inf = 0.1826  # s/10k

# Stacking: 3 models × ~5 MB + meta-learner
stack_size = single_size * 3 + 0.1  # approximate
stack_inf = stacking_robust.get('train_time', 0) / 10  # rough estimate

print(f'{"Metric":<25} | {"Single XGBoost":<18} | {"Stacking":<18}')
print('-'*65)
print(f'{"Model Size (MB)":<25} | {single_size:<18.2f} | {stack_size:<18.2f}')
print(f'{"Inference/10k (s)":<25} | {single_inf:<18.4f} | {"~" + str(round(stack_inf, 3)):<18}')
print(f'{"# Base Models":<25} | {"1":<18} | {"3 + meta-learner":<18}')
print(f'{"MCC (S4 Robust+Adv)":<25} | {s_s4:<18.4f} | {st_s4:<18.4f}')
print('='*65)

## 5. Final Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: S1-S4 MCC comparison
scenarios = ['S1', 'S2', 'S3', 'S4']
x = np.arange(4)
w = 0.35
axes[0].bar(x - w/2, [s_s1, s_s2, s_s3, s_s4], w, label='Single XGBoost', color='steelblue')
axes[0].bar(x + w/2, [st_s1, st_s2, st_s3, st_s4], w, label='Stacking', color='darkorange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(scenarios)
axes[0].set_ylabel('MCC')
axes[0].set_title('S1-S4: Single vs Stacking')
axes[0].legend()
axes[0].set_ylim(-0.1, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Robustness Ablation MCC S4
cfg_labels = [r['config'] for r in stack_rob]
mcc_s4_stack = [r['mcc_s4'] for r in stack_rob]
x2 = np.arange(len(cfg_labels))
axes[1].bar(x2, mcc_s4_stack, color='darkorange', alpha=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(cfg_labels, fontsize=9)
axes[1].set_ylabel('MCC (S4)')
axes[1].set_title('Stacking Robustness per Config')
axes[1].set_ylim(0, 1.1)
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: Efficiency trade-off
models_eff = ['Single\nXGBoost', 'Stacking\nEnsemble']
sizes = [single_size, stack_size]
mccs = [s_s4, st_s4]
colors = ['steelblue', 'darkorange']
axes[2].scatter(sizes, mccs, s=200, c=colors, zorder=5)
for i, txt in enumerate(models_eff):
    axes[2].annotate(txt, (sizes[i], mccs[i]), textcoords='offset points',
                     xytext=(10, -10), fontsize=10)
axes[2].set_xlabel('Model Size (MB)')
axes[2].set_ylabel('MCC (S4 Robust)')
axes[2].set_title('Efficiency vs Robustness')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stacking_final_evaluation.png'), bbox_inches='tight')
plt.show()
print('Saved: stacking_final_evaluation.png')

## 6. Kesimpulan

In [ ]:
print('='*70)
print('  KESIMPULAN AKHIR')
print('='*70)
print(f"""
1. PERFORMA CLEAN (S1):
   - Single XGBoost: MCC = {s_s1:.4f}
   - Stacking Ensemble: MCC = {st_s1:.4f}
   → {'Stacking lebih baik' if st_s1 > s_s1 else 'Single sudah cukup'} pada traffic normal

2. VULNERABILITY (S2):
   - Single: MCC = {s_s2:.4f}
   - Stacking: MCC = {st_s2:.4f}
   → {'Stacking lebih tahan' if st_s2 > s_s2 else 'Keduanya rentan'} terhadap evasion

3. RECOVERY SETELAH AT (S4):
   - Single: MCC = {s_s4:.4f}
   - Stacking: MCC = {st_s4:.4f}
   → {'Stacking recovery lebih baik' if st_s4 > s_s4 else 'Single recovery lebih baik'}

4. EFFICIENCY TRADE-OFF:
   - Single: {single_size:.2f} MB
   - Stacking: ~{stack_size:.2f} MB ({stack_size/single_size:.1f}x lebih besar)
   → Stacking memberikan {'keuntungan robustness' if st_s4 > s_s4 else 'tidak cukup keuntungan'}
     dengan biaya {stack_size/single_size:.1f}x model size

5. REKOMENDASI:
   - Jika prioritas EFISIENSI: Single XGBoost + AT (Top-10)
   - Jika prioritas ROBUSTNESS: Stacking Ensemble + AT
   - Sweet spot: {'Stacking Top-10' if st_s4 > s_s4 else 'Single XGBoost Top-10'}
""")
print('='*70)
print('\nSemua notebook (15-20) selesai.')